# Evaluation system for SGR recommendations engine

The evaluation uses the external Steam recommendations dataset, prepared by `src/data_analysis/system_evaluation/prepare_evaluation_data.ipynb`. The prepared file `data/external/recommendations_ready.json` maps each user to the games they marked as recommended, split into an input history and a held-out target:

```json
{"<user_id>": {"history": [app_id, app_id, ...], "target": app_id}, ...}
```

- `history`: the user's liked games without the most recent one, oldest first (3 to 9 games)
- `target`: the most recent liked game

The dataset contains only users with 4 to 10 liked games (both inclusive), and only games that have a feature vector in `data/parquet/games_fully_vectorized.parquet`. That means the recommender can score every game in both `history` and `target`.

## Evaluation method

### Protocol: chronological leave-one-out

For each evaluated user:

1. Take the user's `history` as input. The most recent game has already been held out as `target`, which follows real use: we predict the next game a user will like from the games they liked before.
2. Pass `history` to the recommender as `{app_id: 10 for app_id in history}`. The dataset only records whether a user liked a game (yes/no), so every game gets the same weight. `recommend()` divides ratings by 10, so each game has weight 1.
3. Ask for the top `K_MAX = 20` recommendations. The recommender already excludes the input games.
4. Record the rank of `target` in that list (1 to 20), or "not found".

Every metric below is computed from these ranks, so the recommender runs only once per user.

### Metrics

| Metric | Definition | Why |
|---|---|---|
| **HR@K** (hit rate) | Share of users whose `target` is in the top K | Main metric: did we suggest the game the user liked next? |
| **MRR@20** (mean reciprocal rank) | Average of `1 / rank`, with 0 when not found | HR treats rank 1 and rank 20 the same; MRR rewards finding the game higher up |
| **Catalog coverage@K** | Share of all games that appear in at least one user's top K | Shows whether the system recommends many different games or the same few to everyone |

HR is reported for K = 5, 10, 15 and 20. K = 15 matches the app, where the API's default `limit` is 15.

### Baselines

A metric on its own can't be interpreted, so the same protocol is run on two baselines:

- **Random:** picks K games at random from the whole catalog (155,353 games). Expected HR@K ≈ K / 155,353, which is about 0.013% for K = 20. This is the floor.
- **Most popular:** recommends the K games liked by the most users, skipping games already in the user's history. Popularity is counted from `history` lists only, never from `target`s, so the baseline doesn't see the answers. This is the baseline the recommender really needs to beat: on Steam data, most held-out games are popular ones.

### Sampling and uncertainty

- Evaluate a random sample of `N_USERS = 50_000` users with a fixed seed (`SEED = 42`), so results can be reproduced. Running all 1.65M users isn't needed for a stable estimate. Each recommender call multiplies the full 155k × 4,100 feature matrix by the profile.
- Report a 95% bootstrap confidence interval for HR@K and MRR (1,000 resamples of users). All methods are scored on the same users, so to compare two of them, bootstrap the *difference* in their scores (a paired bootstrap): if its interval excludes 0, the difference is real.
- For speed, user profiles can be scored in batches (`profiles @ matrix.T`). The batched version must reproduce `recommend()` exactly: the same normalisation, the same weighting and the same exclusion of input games. Check this on a few users before running the full sample.

### Breakdowns

Report HR@20 for the recommender and the popularity baseline separately by:

- **History length** (3 to 9 games): does more input help, or does averaging more games blur the profile?
- **Target popularity** (quartiles of users per game): does the recommender only find well-known games, or does it do better than popularity on niche games? A content-based recommender is often expected to help with the long tail, even if it loses on overall HR.

### Known limitations

- **Positive-only data:** a game the user never reviewed isn't necessarily one they dislike, so a recommendation counted as a miss may still be a good one. HR is a lower bound on real quality and is most useful for comparing variants and baselines, not as an absolute score.
- **Easy hits from franchises and editions:** a target like *Dark Souls III* is easy to find when *Dark Souls II* is in the history. Such hits inflate HR compared with genuine discovery.
- **Time leakage in features:** game features (e.g. `recommendations`, the review count) reflect the time the data was collected, not the time of the review. Recommendations can also include games released after the `target` was reviewed. The app behaves the same way, so this is accepted, but it means results aren't a strict "what we would have known then" test.
- **Only one target per user:** each user contributes a single held-out game, so per-user results are noisy. The sample size and bootstrap intervals account for this.

## Prototype: evaluation on 2,000 users

A first, simple version of the protocol above: HR@K for the SGR recommender and two baselines (random and most popular) on a sample of `N_USERS` users. MRR, coverage, confidence intervals and breakdowns are left for later.

### 1. Load the data and sample users

In [1]:
import json
import random
import sys
from collections import Counter
from pathlib import Path

import pandas as pd

sys.path.insert(0, "../../backend")
from cosine_similarity import recommend

DATA_PATH = Path("../../../data/external/recommendations_ready.json")
VECTORS_PATH = Path("../../../data/parquet/games_fully_vectorized.parquet")
REPORT_PATH = Path("evaluation_report.txt")

N_USERS = 2000
SEED = 50
KS = [5, 10, 15, 20]
K_MAX = max(KS)

with open(DATA_PATH) as f:
    users = json.load(f)

random.seed(SEED)
sample = random.sample(list(users), N_USERS)
print(f"{len(users):,} users in the dataset, {N_USERS:,} sampled")

1,652,783 users in the dataset, 2,000 sampled


### 2. Recommenders

Each method takes a user's `history` and returns `K_MAX` app ids, best first, never including the history games.

- **SGR:** the real recommender from `src/backend/cosine_similarity.py`. Every history game gets rating 10, i.e. the same weight.
- **Most popular:** the games that appear in the most histories across the whole dataset. Targets are not counted, so the baseline can't see the answers.
- **Random:** `K_MAX` games picked at random from the full catalog (every game with a feature vector).

In [2]:
def sgr(history):
    recs = recommend({app_id: 10 for app_id in history}, limit=K_MAX)
    return [r["appid"] for r in recs]


popularity = Counter(app_id for user in users.values() for app_id in user["history"])
most_popular = [app_id for app_id, _ in popularity.most_common(K_MAX + 9)]  # a history has at most 9 games

def popular(history):
    return [app_id for app_id in most_popular if app_id not in history][:K_MAX]


catalog = pd.read_parquet(VECTORS_PATH, columns=["appid"])["appid"].tolist()

def random_pick(history):
    picks = random.sample(catalog, K_MAX + len(history))
    return [app_id for app_id in picks if app_id not in history][:K_MAX]


methods = {"SGR": sgr, "Most popular": popular, "Random": random_pick}

### 3. Evaluate

For every sampled user and every method, a hit at K means the `target` is in the top K recommendations. HR@K is the share of users with a hit.

The first SGR call loads the feature matrix (~2.5 GB), so it takes a while.

In [3]:
hits = {name: {k: 0 for k in KS} for name in methods}

for user_id in sample:
    history = users[user_id]["history"]
    target = users[user_id]["target"]
    for name, method in methods.items():
        recs = method(history)
        for k in KS:
            if target in recs[:k]:
                hits[name][k] += 1

hr = {name: {k: hits[name][k] / N_USERS for k in KS} for name in methods}
pd.DataFrame(hr).rename(index=lambda k: f"HR@{k}").map(lambda x: f"{x:.2%}")

,SGR,Most popular,Random
HR@5,1.10%,1.80%,0.00%
HR@10,1.55%,3.75%,0.00%
HR@15,1.90%,5.20%,0.00%
HR@20,2.35%,6.85%,0.00%


### 4. Report

The report compares SGR with both baselines as a ratio (*lift*): "3.0x" means SGR finds the target 3 times as often.

With 2,000 users the random baseline almost never hits, so its measured HR is usually 0 and a ratio can't be computed. SGR is therefore compared with the **expected** random hit rate, `K / catalog size`. The measured random HR is still shown as a sanity check.

In [4]:
expected_random = {k: k / len(catalog) for k in KS}

def lift(a, b):
    return f"{a / b:,.1f}x" if b > 0 else "n/a (0 hits)"

lines = [
    "SGR recommender evaluation - prototype",
    "",
    f"users evaluated:  {N_USERS:,} (random sample, seed {SEED}, of {len(users):,})",
    f"catalog size:     {len(catalog):,} games",
    "method:           chronological leave-one-out; the user's history is the input,",
    "                  a hit means the held-out most recent game is in the top K",
    "",
    "Hit rate (HR@K)",
    f"{'':8}{'SGR':>10}{'Most popular':>15}{'Random':>10}{'Random (expected)':>20}",
]
for k in KS:
    lines.append(
        f"{f'HR@{k}':8}{hr['SGR'][k]:>10.2%}{hr['Most popular'][k]:>15.2%}"
        f"{hr['Random'][k]:>10.2%}{expected_random[k]:>20.4%}"
    )

lines += ["", "SGR compared with baselines (SGR HR / baseline HR)",
          f"{'':8}{'vs random (expected)':>22}{'vs most popular':>18}"]
for k in KS:
    lines.append(
        f"{f'HR@{k}':8}{lift(hr['SGR'][k], expected_random[k]):>22}"
        f"{lift(hr['SGR'][k], hr['Most popular'][k]):>18}"
    )

lines += ["", "Hits (users with the target in the top K)",
          f"{'':8}{'SGR':>10}{'Most popular':>15}{'Random':>10}"]
for k in KS:
    lines.append(f"{f'HR@{k}':8}{hits['SGR'][k]:>10}{hits['Most popular'][k]:>15}{hits['Random'][k]:>10}")

report = "\n".join(lines) + "\n"
REPORT_PATH.write_text(report)
print(report)
print(f"saved to {REPORT_PATH.resolve()}")

SGR recommender evaluation - prototype

users evaluated:  2,000 (random sample, seed 50, of 1,652,783)
catalog size:     155,353 games
method:           chronological leave-one-out; the user's history is the input,
                  a hit means the held-out most recent game is in the top K

Hit rate (HR@K)
               SGR   Most popular    Random   Random (expected)
HR@5         1.10%          1.80%     0.00%             0.0032%
HR@10        1.55%          3.75%     0.00%             0.0064%
HR@15        1.90%          5.20%     0.00%             0.0097%
HR@20        2.35%          6.85%     0.00%             0.0129%

SGR compared with baselines (SGR HR / baseline HR)
          vs random (expected)   vs most popular
HR@5                    341.8x              0.6x
HR@10                   240.8x              0.4x
HR@15                   196.8x              0.4x
HR@20                   182.5x              0.3x

Hits (users with the target in the top K)
               SGR   Most popula

### 5. Plot: hit rate by K

HR@K for each method as K grows, using the results from step 3. The random baseline is drawn as a dashed line because it shows the expected value (`K / catalog size`), not a measured one. On this scale it stays close to 0, which is the point: it shows how far both real methods are above chance.

In [ ]:
import matplotlib.pyplot as plt

series = {
    "SGR": ([hr["SGR"][k] for k in KS], "#2a78d6", "-"),
    "Most popular": ([hr["Most popular"][k] for k in KS], "#eb6834", "-"),
    "Random (expected)": ([expected_random[k] for k in KS], "#1baf7a", "--"),
}

fig, ax = plt.subplots(figsize=(9, 4.5))
for name, (values, color, style) in series.items():
    ax.plot(KS, [v * 100 for v in values], style, color=color, linewidth=2,
            marker="o", markersize=7, markeredgecolor="white", markeredgewidth=2, label=name)
    # direct label at the end of each line, in text color
    ax.annotate(f"{name}  {values[-1]:.2%}", xy=(KS[-1], values[-1] * 100),
                xytext=(10, 0), textcoords="offset points", va="center", color="#0b0b0b", fontsize=10)

ax.set_title("Hit rate by number of recommendations (K)", loc="left", fontsize=13, color="#0b0b0b")
ax.set_xlabel("K (number of recommended games)", color="#52514e")
ax.set_ylabel("HR@K (%)", color="#52514e")
ax.set_xticks(KS)
ax.set_xlim(KS[0] - 1, KS[-1] + 7)  # room for the direct labels
ax.set_ylim(bottom=0)
ax.grid(axis="y", color="#e8e7e2", linewidth=1)
ax.set_axisbelow(True)
ax.legend(frameon=False, loc="upper left", labelcolor="#52514e")
ax.tick_params(colors="#52514e", length=0)
for side in ["top", "right", "left"]:
    ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color("#d8d7d2")
plt.tight_layout()
plt.show()

## Summary of the prototype results

Results for the sample with `SEED = 50` (2,000 users):

| | SGR | Most popular | Random (expected) |
|---|---|---|---|
| HR@5 | 1.10% | 1.80% | 0.003% |
| HR@10 | 1.55% | 3.75% | 0.006% |
| HR@15 | 1.90% | 5.20% | 0.010% |
| HR@20 | 2.35% | 6.85% | 0.013% |

### What the plot shows

- **Most popular is the highest line at every K** and rises quickly, from 1.80% at K = 5 to 6.85% at K = 20.
- **SGR is clearly above random but well below most popular.** It rises slowly, from 1.10% to 2.35%.
- **Random stays at almost 0.** At K = 20 the expected hit rate is 0.013%, so on this scale its line lies on the axis.
- **The gap between SGR and most popular grows with K.** Popularity gains about 1.3 points per 5 extra games, while SGR gains about 0.4.

### How we interpret it

- **SGR's vectors carry real signal.** It finds the user's next game about 180 times as often as random guessing does at K = 20 (2.35% vs 0.013%). The feature vectors capture something about what users like.
- **The task favours popular games.** A small set of very popular games covers a large share of what users like next: across the whole dataset, 6.5% of the target games are among the top 20 most popular games, and 23.4% are among the top 100. Recommending bestsellers is therefore a strong, easy guess.
- **SGR ignores popularity.** It recommends games *similar* to the history, often niche games with matching tags and descriptions, while users often go on to like a hit. That's why it finds the target only about a third as often as the popularity baseline at K = 20.
- **Hit rate doesn't measure everything.** The popularity baseline gives almost every user the same list, which isn't personalised and doesn't help users discover new games. Losing to it on HR doesn't make SGR useless, but it shows that adding popularity to SGR could help. That is tested in the experiment below.
- **Treat the numbers as rough.** At N = 2,000, SGR has only 47 hits at K = 20. With `SEED = 42`, SGR's HR@20 was 1.65% and popularity's was 7.85%. Differences under about 1 point between samples are noise, but "SGR ≫ random" and "most popular > SGR" hold on both samples.

## Experiment: SGR with a popularity boost

Because most popular beats SGR, we tested whether mixing popularity into SGR's scores helps. Each game's score becomes:

```
score = cosine similarity + ALPHA * popularity
```

- **cosine similarity:** exactly what `recommend()` computes, between the user's profile and the game
- **popularity:** `log(1 + number of users who liked the game)`, scaled to 0–1 so the most popular game gets 1. The log stops a few huge hits from taking over. Counts come from `history` lists only, as in the popularity baseline.
- **ALPHA:** how strong the boost is. `ALPHA = 0` is plain SGR, and a very large `ALPHA` turns into the most-popular baseline.

This lives only in this notebook; the backend is unchanged. The code reuses the feature matrix that `cosine_similarity.py` loads, and computes the same profile as `recommend()`. Every history game has the same weight, so the weighted average is a plain mean.

Instead of building a top-20 list for every `ALPHA`, the code computes the target's **rank** directly: 1 + the number of games that score higher than the target. A hit at K means rank ≤ K. Similarities are computed once per user and reused for every `ALPHA`.

In [ ]:
import numpy as np
import cosine_similarity as cs

ALPHAS = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]

cs._load_data_if_needed()

# popularity of every game in the matrix, in the matrix's row order (0 for games nobody liked)
pop_scores = np.zeros(len(cs._appids), dtype=np.float32)
for app_id, count in popularity.items():
    pop_scores[cs._appid_to_index[app_id]] = count
pop_scores = np.log1p(pop_scores) / np.log1p(pop_scores.max())


def similarities(history):
    rows = [cs._appid_to_index[app_id] for app_id in history]
    profile = cs._matrix[rows].mean(axis=0)
    profile /= np.linalg.norm(profile)
    sims = cs._matrix @ profile
    sims[rows] = -np.inf  # never recommend the user's own games
    return sims


boost_hits = {alpha: {k: 0 for k in KS} for alpha in ALPHAS}

for user_id in sample:
    history = users[user_id]["history"]
    target_row = cs._appid_to_index[users[user_id]["target"]]
    sims = similarities(history)
    for alpha in ALPHAS:
        scores = sims + alpha * pop_scores
        rank = 1 + (scores > scores[target_row]).sum()
        for k in KS:
            if rank <= k:
                boost_hits[alpha][k] += 1

boost_hr = pd.DataFrame(
    {f"SGR, ALPHA={alpha}": {f"HR@{k}": boost_hits[alpha][k] / N_USERS for k in KS} for alpha in ALPHAS}
)
boost_hr["Most popular"] = {f"HR@{k}": hr["Most popular"][k] for k in KS}
boost_hr.map(lambda x: f"{x:.2%}")

`ALPHA = 0` should match the SGR results from step 3; it's a check that this code computes the same thing as `recommend()`. Small differences can come from ties in similarity.

The table is also appended to the report file.

In [ ]:
boost_lines = ["", "Experiment: SGR + ALPHA * popularity (HR@K)",
               f"{'':8}" + "".join(f"{f'a={alpha}':>9}" for alpha in ALPHAS) + f"{'Popular':>10}"]
for k in KS:
    row = "".join(f"{boost_hits[alpha][k] / N_USERS:>9.2%}" for alpha in ALPHAS)
    boost_lines.append(f"{f'HR@{k}':8}{row}{hr['Most popular'][k]:>10.2%}")

boost_report = "\n".join(boost_lines) + "\n"
with open(REPORT_PATH, "a") as f:
    f.write(boost_report)
print(boost_report)

### Results of the first run (ALPHA from 0 to 0.5)

These results come from an earlier run with `ALPHAS = [0, 0.02, 0.05, 0.1, 0.2, 0.3, 0.5]`. The code above now uses `ALPHAS = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]`, so its output table will differ. Update this section after running it.

The experiment used the same 2,000 users and protocol as above (`SEED = 50`). The backend was not changed. `ALPHA = 0` reproduced SGR's results exactly (47 hits at K = 20), which confirms the experiment computed the same scores as `recommend()`.

| | ALPHA = 0 (SGR) | 0.02 | 0.05 | 0.1 | 0.2 | 0.3 | 0.5 | Most popular |
|---|---|---|---|---|---|---|---|---|
| HR@5 | 1.10% | 1.50% | 1.90% | 2.10% | 2.60% | 2.65% | **2.85%** | 1.80% |
| HR@10 | 1.55% | 2.05% | 2.60% | 3.35% | 4.15% | 4.50% | **5.05%** | 3.75% |
| HR@15 | 1.90% | 2.75% | 3.10% | 4.15% | 5.50% | 6.45% | **7.00%** | 5.20% |
| HR@20 | 2.35% | 3.15% | 3.75% | 4.80% | 6.55% | 7.30% | **8.30%** | 6.85% |

### Interpretation

- **The boost helps a lot.** At `ALPHA = 0.5`, HR@20 goes from 2.35% to 8.30%, about 3.5 times higher.
- **SGR with the boost beats both methods on their own.** At `ALPHA = 0.5` it is above most popular at every K. Similarity and popularity carry different information, and combining them works better than either one alone.
- **The best `ALPHA` is not found yet.** HR still rises at 0.5, so larger values (e.g. 1.0 and 2.0) should be tested. At some point it has to fall back towards the popularity baseline.

### Before using it in the backend

- **Confirm `ALPHA` on another sample.** `ALPHA` was chosen on the same users it was scored on, which makes the best value look slightly better than it is. Pick `ALPHA` on one sample (e.g. `SEED = 50`) and confirm it on another (e.g. `SEED = 42`).
- **Measure what it costs.** A higher `ALPHA` pushes the same popular games to everyone, so recommendations become less personal and cover fewer games. HR doesn't show this. Catalog coverage and the average popularity of recommended games would, and the final `ALPHA` should balance hit rate against them.